In [1]:
from bs4 import BeautifulSoup
import urllib.request
import urllib.error
import json
from urllib.parse import urljoin

**Robots**

In [2]:
def fetch_html(page_url):
    try:
        response = urllib.request.urlopen(page_url)
        return response.read().decode("utf-8", errors="ignore")
    except urllib.error.HTTPError as e:
        print(f"HTTP Error {e.code}: {e.reason}")
    except urllib.error.URLError as e:
        print(f"Network Error: {e.reason}")
    except Exception as e:
        print(f"Unexpected error: {e}")
    return None

In [3]:
url = "http://mlg.ucd.ie/modules/python/sources/rental/index.html"
robots_url = "http://mlg.ucd.ie/robots.txt"
print(fetch_html(robots_url))

User-agent: *
disallow: /files/*

User-agent: *
disallow: /datasets/comp30120/*

User-agent: *
disallow: /datasets/comp41450/*

User-agent: *
disallow: /datasets/comp41680/*



talk about robots

In [4]:
def value_for(table, label):
    label_cell = table.find("td", string=lambda s: s and s.strip().startswith(label))
    if not label_cell:
        return None
    value_cell = label_cell.find_next_sibling("td")
    return value_cell.get_text(separator=" ", strip=True) if value_cell else None

travelled_to = set()
queue = [url]
records = []

classes = [("price", "Price:"),("location", "Location:"),("bedrooms", "Bedrooms:"),("bathrooms", "Bathrooms:"),("parking", "Parking:"),("garden", "Garden:"),("lease_length", "Lease Length:"),("contact", "Contact:"),]
while queue:
    current_url = queue.pop(0)
    if current_url in travelled_to:
        continue

    html = fetch_html(current_url)
    travelled_to.add(current_url)

    soup = BeautifulSoup(html, "html.parser")
    transactions = []

    for table in soup.select("table.listing-table"):
        header_el = table.find_parent("li").select_one("span.record")
        record = {"header": header_el.get_text(strip=True)}

        for key, label in classes:
            record[key] = value_for(table, label)
        transactions.append(record)
    records.append(transactions)

    for a in soup.find_all("a", href=True):
        link = urljoin(current_url, a["href"])
        if link not in travelled_to:
            queue.append(link)

In [5]:
with open("./transactions.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)